# Классификация глазных заболеваний
**Курс:** Основы теории принятия решений

### Порядок запуска
Выполняй ячейки **строго по порядку**. Ячейки 1–6 создают все файлы проекта — запусти их один раз в начале каждой сессии.

In [4]:
# ── Ячейка 1: Установка библиотек ─────────────────────────────────────────
!pip install torch torchvision scikit-learn matplotlib seaborn kaggle -q
import sys, os
sys.path.insert(0, ".")
os.makedirs("src", exist_ok=True)
os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/confusion_matrices", exist_ok=True)
print("OK")

OK


In [ ]:
# ── Ячейка 2: Kaggle API ───────────────────────────────────────────────────
import os
os.makedirs("/root/.kaggle", exist_ok=True)

# Вставь свой токен сюда (Kaggle → Settings → API → Create New Token)
KAGGLE_TOKEN = "KGAT_ВСТАВЬ_СВОЙ_ТОКЕН_СЮДА"

with open("/root/.kaggle/access_token", "w") as f:
    f.write(KAGGLE_TOKEN)
!chmod 600 /root/.kaggle/access_token
print("Kaggle настроен")

In [ ]:
# ── Ячейка 3: Создаём config.py ───────────────────────────────────────────

In [6]:
%%writefile config.py
from dataclasses import dataclass, field
import torch


@dataclass
class ExperimentConfig:
    # --- Данные ---
    kaggle_dataset: str              = "gunavenkatdoddi/eye-diseases-classification"
    raw_data_dir: str                = "data/raw"
    img_sizes: tuple                 = (128,)
    train_ratio: float               = 0.8
    batch_size: int                  = 64
    num_classes: int                 = 4

    # --- Шум ---
    snr_levels_db: tuple             = (20.0, 15.0, 10.0, 5.0, 0.0, -5.0)

    # --- Монте-Карло ---
    n_repetitions: int               = 15
    train_fractions: tuple           = (0.25, 0.50, 1.0)
    fixed_snr_for_samples_exp: float = 10.0

    # --- CNN ---
    num_epochs: int      = 20
    learning_rate: float = 1e-3

    # --- Пути ---
    results_dir: str = "results"
    drive_dir: str   = "/content/drive/MyDrive/eye_classification"

    device: str = field(default="")

    def __post_init__(self) -> None:
        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

Overwriting config.py


In [8]:
%%writefile src/__init__.py


Overwriting src/__init__.py


In [7]:
%%writefile src/noise.py
import math
import torch
from torch import Tensor


class GaussianNoise:
    """
    Аддитивный белый гауссовский шум (AWGN).

    Модель сигнала: y = x + n,  n ~ N(0, σ²·I)

    SNR в дБ: SNR_dB = 10·log₁₀(P_signal / σ²),  P_signal = E[x²]
    Отсюда:   σ = √(P_signal / 10^(SNR_dB/10))
    """

    @staticmethod
    def snr_to_sigma(images: Tensor, snr_db: float) -> float:
        """Вычисляет σ шума из мощности сигнала и заданного SNR."""
        p_signal = images.pow(2).mean().item()
        if p_signal == 0:
            return 0.0
        return math.sqrt(p_signal / (10 ** (snr_db / 10)))

    @staticmethod
    def add(images: Tensor, snr_db: float) -> Tensor:
        """Возвращает зашумлённую копию батча. Оригинал не изменяется."""
        sigma = GaussianNoise.snr_to_sigma(images, snr_db)
        return images + torch.randn_like(images) * sigma

Overwriting src/noise.py


In [9]:
%%writefile src/optimal_model.py
import torch
from torch import Tensor


class NearestCentroidClassifier:
    """
    Оптимальный байесовский классификатор для модели AWGN.

    При y = x_c + n, n ~ N(0, σ²·I), оптимальное решение:
        ĉ = argmin_c ||y - x_c||²
    — минимум евклидова расстояния до эталона класса.
    """

    def __init__(self) -> None:
        self.centroids = None  # (n_classes, C, H, W)

    def fit(self, centroids: Tensor) -> "NearestCentroidClassifier":
        self.centroids = centroids
        return self

    def predict(self, images: Tensor) -> Tensor:
        """images: (B, C, H, W) → preds: (B,)"""
        if self.centroids is None:
            raise RuntimeError("Вызовите fit() перед predict().")

        device    = images.device
        centroids = self.centroids.to(device)
        flat      = images.view(images.shape[0], -1)           # (B, D)
        ctrs      = centroids.view(centroids.shape[0], -1)     # (n_classes, D)

        # ||a-b||² = ||a||² + ||b||² - 2aᵀb
        dists = (
            flat.pow(2).sum(1, keepdim=True)
            + ctrs.pow(2).sum(1).unsqueeze(0)
            - 2 * flat @ ctrs.T
        )
        return dists.argmin(dim=1)

Overwriting src/optimal_model.py


In [10]:
%%writefile src/cnn_model.py
import torch.nn as nn
from torch import Tensor


class EyeCNN(nn.Module):
    """
    4 блока Conv→BN→ReLU→Pool + AdaptiveAvgPool→Dropout→Linear.
    AdaptiveAvgPool позволяет использовать одну архитектуру для 32×32 и 128×128.
    """

    def __init__(self, img_size: int, num_classes: int) -> None:
        super().__init__()
        self.img_size = img_size

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),  # (B,256,H,W) → (B,256,1,1) для любого H,W
        )
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(256, num_classes))

    def forward(self, x: Tensor) -> Tensor:
        return self.head(self.features(x).flatten(1))

Overwriting src/cnn_model.py


In [20]:
%%writefile src/data_loader.py
import os
from collections import defaultdict
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder

from config import ExperimentConfig


class EyeDataset:
    """Загружает датасет с Kaggle, возвращает DataLoader'ы и центроиды классов."""

    _MEAN = (0.485, 0.456, 0.406)
    _STD  = (0.229, 0.224, 0.225)

    def __init__(self, config: ExperimentConfig) -> None:
        self.config      = config
        self.data_dir    = Path(config.raw_data_dir)
        self.class_names: list = []

    def download(self) -> None:
        """Скачивает датасет с Kaggle если данные ещё не существуют."""
        if self.data_dir.exists() and any(self.data_dir.iterdir()):
            print("Dataset already present, skipping download.")
            return
        self.data_dir.mkdir(parents=True, exist_ok=True)
        os.system(
            f"kaggle datasets download -d {self.config.kaggle_dataset} "
            f"-p {self.data_dir} --unzip"
        )

    def get_loaders(self, img_size: int, train_fraction: float = 1.0):
        """
        Возвращает (train_loader, test_loader).
        Датасет без готового train/test сплита — делаем стратифицированно 80/20.
        """
        root    = self._find_dataset_root()
        full_ds = ImageFolder(root, transform=self._test_tf(img_size))
        self.class_names = full_ds.classes

        train_idx, test_idx = self._stratified_split(full_ds, self.config.train_ratio)

        if train_fraction < 1.0:
            n         = int(len(train_idx) * train_fraction)
            train_idx = train_idx[:n]

        # Тренировочный сплит — с аугментациями, тестовый — без
        train_ds_aug = ImageFolder(root, transform=self._train_tf(img_size))
        train_ds = Subset(train_ds_aug, train_idx)
        test_ds  = Subset(full_ds,      test_idx)

        kw = dict(batch_size=self.config.batch_size, num_workers=0,
                  pin_memory=(self.config.device == "cuda"))
        return (DataLoader(train_ds, shuffle=True,  **kw),
                DataLoader(test_ds,  shuffle=False, **kw))

    def compute_centroids(self, loader: DataLoader) -> torch.Tensor:
        """Среднее изображение каждого класса — эталон для оптимального классификатора."""
        device = torch.device(self.config.device)
        sums, counts = None, None

        for images, labels in loader:
            images = images.to(device)
            n_cls  = int(labels.max().item()) + 1
            if sums is None:
                sums   = torch.zeros(n_cls, *images.shape[1:], device=device)
                counts = torch.zeros(n_cls, device=device)
            for c in range(n_cls):
                mask = labels == c
                if mask.any():
                    sums[c]   += images[mask].sum(0)
                    counts[c] += mask.sum()

        return sums / counts.view(-1, 1, 1, 1)

    def _train_tf(self, s: int) -> transforms.Compose:
        return transforms.Compose([
            transforms.Resize((s, s)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(self._MEAN, self._STD),
        ])

    def _test_tf(self, s: int) -> transforms.Compose:
        # Тестовые изображения — без аугментации, только resize и нормализация
        return transforms.Compose([
            transforms.Resize((s, s)),
            transforms.ToTensor(),
            transforms.Normalize(self._MEAN, self._STD),
        ])

    def _stratified_split(self, dataset: ImageFolder, train_ratio: float):
        """
        Стратифицированный сплит: пропорции классов одинаковы в train и test.
        Случайное перемешивание внутри каждого класса — основа метода Монте-Карло.
        """
        class_indices = defaultdict(list)
        for idx, (_, label) in enumerate(dataset.samples):
            class_indices[label].append(idx)

        train_idx, test_idx = [], []
        for indices in class_indices.values():
            perm  = torch.randperm(len(indices)).tolist()
            split = int(len(indices) * train_ratio)
            train_idx.extend([indices[i] for i in perm[:split]])
            test_idx.extend( [indices[i] for i in perm[split:]])

        return train_idx, test_idx

    def _find_dataset_root(self) -> Path:
        """
        Находит папку с подпапками классов.
        Поддерживает структуру: data/raw/dataset/{class}/ или data/raw/{class}/.
        """
        if not self.data_dir.exists():
            raise FileNotFoundError(f"{self.data_dir} не найдена. Запустите download().")

        candidates = [self.data_dir, *(p for p in self.data_dir.iterdir() if p.is_dir())]
        for candidate in candidates:
            subdirs = [p for p in candidate.iterdir() if p.is_dir()]
            # Папка с классами: все поддиректории содержат изображения
            if subdirs and all(
                any(f.suffix.lower() in (".jpg", ".jpeg", ".png") for f in d.iterdir())
                for d in subdirs
            ):
                return candidate

        raise FileNotFoundError(f"Папки классов не найдены в {self.data_dir}.")

Overwriting src/data_loader.py


In [21]:
%%writefile src/trainer.py
import torch
import torch.nn as nn
from torch import Tensor
from torch.utils.data import DataLoader

from config import ExperimentConfig
from src.cnn_model import EyeCNN
from src.noise import GaussianNoise
from src.optimal_model import NearestCentroidClassifier


class Trainer:
    """Обучает CNN и оптимальный классификатор, оценивает точность на зашумлённых данных."""

    def __init__(self, config: ExperimentConfig) -> None:
        self.config = config
        self.device = torch.device(config.device)

    def train_cnn(self, model: EyeCNN, loader: DataLoader) -> EyeCNN:
        """Обучение на чистых данных. Шум добавляется только при evaluate()."""
        model     = model.to(self.device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=self.config.learning_rate)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

        model.train()
        for epoch in range(self.config.num_epochs):
            for images, labels in loader:
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                criterion(model(images), labels).backward()
                optimizer.step()
            scheduler.step()
            print(f"    Эпоха {epoch+1}/{self.config.num_epochs}", end="\r")
        return model

    def fit_optimal(self, centroids: Tensor) -> NearestCentroidClassifier:
        return NearestCentroidClassifier().fit(centroids)

    @torch.no_grad()
    def evaluate(self, model, loader: DataLoader, snr_db: float) -> float:
        """Точность на тест-данных с добавленным гауссовским шумом."""
        is_cnn = isinstance(model, EyeCNN)
        if is_cnn:
            model.eval()

        correct = total = 0
        for images, labels in loader:
            images = images.to(self.device)
            noisy  = GaussianNoise.add(images, snr_db)
            preds  = model(noisy).argmax(1) if is_cnn else model.predict(noisy)
            correct += (preds.cpu() == labels).sum().item()
            total   += len(labels)

        return correct / total

Overwriting src/trainer.py


In [22]:
%%writefile src/experiment.py
import numpy as np

from config import ExperimentConfig
from src.cnn_model import EyeCNN
from src.data_loader import EyeDataset
from src.trainer import Trainer


class MonteCarloRunner:
    """
    Метод Монте-Карло: для каждого повторения — новый случайный сплит,
    обучение обоих классификаторов, оценка при каждом уровне SNR.
    Результат: матрица точностей (n_snr × n_repetitions).
    """

    def __init__(self, config: ExperimentConfig) -> None:
        self.config  = config
        self.dataset = EyeDataset(config)
        self.trainer = Trainer(config)

    def run(self, img_size: int, train_fraction: float = 1.0) -> dict:
        cfg    = self.config
        n_snr  = len(cfg.snr_levels_db)
        results = {"cnn": np.zeros((n_snr, cfg.n_repetitions)),
                   "optimal": np.zeros((n_snr, cfg.n_repetitions))}

        for rep in range(cfg.n_repetitions):
            print(f"  [{img_size}px {train_fraction:.0%}] Повторение {rep+1}/{cfg.n_repetitions}", end="\r")

            train_loader, test_loader = self.dataset.get_loaders(img_size, train_fraction)
            centroids = self.dataset.compute_centroids(train_loader)

            cnn     = self.trainer.train_cnn(EyeCNN(img_size, cfg.num_classes), train_loader)
            optimal = self.trainer.fit_optimal(centroids)

            for i, snr_db in enumerate(cfg.snr_levels_db):
                results["cnn"][i, rep]     = self.trainer.evaluate(cnn,     test_loader, snr_db)
                results["optimal"][i, rep] = self.trainer.evaluate(optimal, test_loader, snr_db)

        print()
        return results

    def run_samples_experiment(self, img_size: int) -> dict:
        """Серия 2: точность vs доля обучающих данных при фиксированном SNR."""
        cfg     = self.config
        snr_idx = list(cfg.snr_levels_db).index(cfg.fixed_snr_for_samples_exp)
        results = {}
        for frac in cfg.train_fractions:
            print(f"\n  Fraction = {frac:.0%}")
            r = self.run(img_size=img_size, train_fraction=frac)
            results[frac] = {"cnn": r["cnn"][snr_idx], "optimal": r["optimal"][snr_idx]}
        return results

Overwriting src/experiment.py


In [23]:
%%writefile src/evaluate.py
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader

from config import ExperimentConfig
from src.cnn_model import EyeCNN
from src.noise import GaussianNoise
from src.optimal_model import NearestCentroidClassifier


class Evaluator:
    """Строит все графики и confusion matrix'ы, сохраняет на диск и в Google Drive."""

    _COLORS = {"cnn": "#2196F3", "optimal": "#FF5722"}
    _LABELS = {"cnn": "CNN (обучение с учителем)", "optimal": "Оптимальный (ближайший центроид)"}

    def __init__(self, config: ExperimentConfig, class_names: list) -> None:
        self.config      = config
        self.class_names = class_names
        self.plots_dir   = Path(config.results_dir) / "plots"
        self.cm_dir      = Path(config.results_dir) / "confusion_matrices"
        self.plots_dir.mkdir(parents=True, exist_ok=True)
        self.cm_dir.mkdir(parents=True, exist_ok=True)

    def accuracy_vs_snr(self, results_by_size: dict) -> None:
        """Точность vs SNR для обоих классификаторов и обоих размеров изображений."""
        n = len(self.config.img_sizes)
        fig, axes = plt.subplots(1, n, figsize=(7*n, 5), sharey=True)
        if n == 1: axes = [axes]
        snr = list(self.config.snr_levels_db)

        for ax, img_size in zip(axes, self.config.img_sizes):
            for name, data in results_by_size[img_size].items():
                mean, std = data.mean(1), data.std(1)
                ax.plot(snr, mean, marker="o", lw=2, label=self._LABELS[name], color=self._COLORS[name])
                ax.fill_between(snr, mean-std, mean+std, alpha=0.2, color=self._COLORS[name])
            ax.set(title=f"{img_size}×{img_size}", xlabel="SNR (дБ)", ylabel="Точность", ylim=(0, 1.05))
            ax.invert_xaxis()
            ax.legend(fontsize=9); ax.grid(alpha=0.4)

        fig.suptitle("Точность классификации vs уровень шума", fontsize=14)
        plt.tight_layout()
        self._save(fig, self.plots_dir / "accuracy_vs_snr.png")

    def accuracy_vs_samples(self, results_by_fraction: dict, img_size: int) -> None:
        """Точность vs доля обучающих данных при фиксированном SNR."""
        fractions = list(results_by_fraction.keys())
        x, w = np.arange(len(fractions)), 0.35
        fig, ax = plt.subplots(figsize=(8, 5))

        for i, name in enumerate(["cnn", "optimal"]):
            means = [results_by_fraction[f][name].mean() for f in fractions]
            stds  = [results_by_fraction[f][name].std()  for f in fractions]
            ax.bar(x + i*w, means, w, label=self._LABELS[name],
                   color=self._COLORS[name], yerr=stds, capsize=5, alpha=0.85)

        ax.set_xticks(x + w/2)
        ax.set_xticklabels([f"{int(f*100)}%" for f in fractions])
        ax.set(xlabel="Объём выборки", ylabel="Точность", ylim=(0, 1.05),
               title=f"Точность vs число выборок (SNR={self.config.fixed_snr_for_samples_exp} дБ, {img_size}×{img_size})")
        ax.legend(); ax.grid(axis="y", alpha=0.4)
        plt.tight_layout()
        self._save(fig, self.plots_dir / f"accuracy_vs_samples_{img_size}.png")

    def plot_confusion_matrix(self, model, loader: DataLoader, snr_db: float, title: str) -> None:
        """Confusion matrix при заданном уровне шума."""
        device = torch.device(self.config.device)
        is_cnn = isinstance(model, EyeCNN)
        if is_cnn: model.eval()
        preds_all, true_all = [], []

        with torch.no_grad():
            for images, labels in loader:
                noisy = GaussianNoise.add(images.to(device), snr_db)
                preds = model(noisy).argmax(1) if is_cnn else model.predict(noisy)
                preds_all.extend(preds.cpu().tolist())
                true_all.extend(labels.tolist())

        cm = confusion_matrix(true_all, preds_all)
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=self.class_names, yticklabels=self.class_names, ax=ax)
        ax.set(xlabel="Предсказание", ylabel="Истина", title=title)
        plt.xticks(rotation=30, ha="right"); plt.tight_layout()
        safe = "".join(c if c.isalnum() or c in "_-" else "_" for c in title)
        self._save(fig, self.cm_dir / f"{safe}.png")

    def save_to_drive(self) -> None:
        drive = Path(self.config.drive_dir)
        if not drive.exists():
            print("Drive не смонтирован — пропускаем."); return
        shutil.copytree(self.config.results_dir, str(drive / "results"), dirs_exist_ok=True)
        print(f"Сохранено в {drive / 'results'}")

    @staticmethod
    def _save(fig, path: Path) -> None:
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.show(); plt.close(fig)
        print(f"  Сохранено: {path}")

Overwriting src/evaluate.py


---
## Запуск экспериментов
Ячейки выше создали все файлы. Дальше — сами эксперименты.

In [15]:
# ── Ячейка 12: Google Drive (для сохранения результатов) ──────────────────
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# ── Ячейка 13: Импорты и конфигурация ─────────────────────────────────────
import torch
from config import ExperimentConfig
from src.data_loader import EyeDataset
from src.experiment import MonteCarloRunner
from src.evaluate import Evaluator
from src.cnn_model import EyeCNN
from src.trainer import Trainer

cfg = ExperimentConfig()
print(f"Device:      {cfg.device}")
print(f"Img sizes:   {cfg.img_sizes}")
print(f"SNR levels:  {cfg.snr_levels_db}")
print(f"Repetitions: {cfg.n_repetitions}")
if cfg.device == "cpu":
    print("⚠️  GPU не найден! Runtime → Change runtime type → T4 GPU")

Device:      cuda
Img sizes:   (128,)
SNR levels:  (20.0, 15.0, 10.0, 5.0, 0.0, -5.0)
Repetitions: 15


In [25]:
# ── Ячейка 14: Загрузка данных ────────────────────────────────────────────
dataset = EyeDataset(cfg)
dataset.download()

train_loader, test_loader = dataset.get_loaders(img_size=128)
images, labels = next(iter(train_loader))
print(f"Классы:       {dataset.class_names}")
print(f"Батч:         {images.shape}")
print(f"Train батчей: {len(train_loader)}")
print(f"Test батчей:  {len(test_loader)}")

Dataset already present, skipping download.


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Классы:       ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']
Батч:         torch.Size([64, 3, 128, 128])
Train батчей: 53
Test батчей:  14


In [26]:
import matplotlib.pyplot as plt
from torchvision import datasets
import numpy as np

dataset = datasets.ImageFolder(root='data/raw/dataset')
class_names = dataset.classes  # ['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']

sample_images = []
sample_titles = []
for class_idx, class_name in enumerate(class_names):
    for img, label in dataset:
        if label == class_idx:
            sample_images.append(np.array(img))
            sample_titles.append(class_name.replace('_', ' ').title())
            break

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, (img, title) in enumerate(zip(sample_images, sample_titles)):
    axes[i].imshow(img)
    axes[i].set_title(title)
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('samples.png', dpi=150, bbox_inches='tight')
plt.show()

KeyboardInterrupt: 

In [ ]:
from torchsummary import summary
from src.cnn_model import EyeCNN

model = EyeCNN(img_size=128, num_classes=4).to(cfg.device)
summary(model, (3, 128, 128))

In [ ]:
# ── Ячейка 15: Серия 1 — Монте-Карло: точность vs SNR ────────────────────
# ⏱ ~1.5 часа на GPU. Для быстрой проверки раскомментируй строки ниже:
# cfg.n_repetitions = 2
# cfg.num_epochs    = 2

runner = MonteCarloRunner(cfg)
results_by_size = {}

for img_size in cfg.img_sizes:
    print(f"\n{'='*40}\nРазмер {img_size}×{img_size}\n{'='*40}")
    results_by_size[img_size] = runner.run(img_size=img_size)

print("\nСерия 1 завершена!")


Размер 128×128


In [ ]:
# ── Ячейка 16: Серия 2 — точность vs число выборок ───────────────────────
# ⏱ ~45 минут
results_by_fraction = runner.run_samples_experiment(img_size=128)
print("\nСерия 2 завершена!")

In [ ]:
# ── Ячейка 17: Графики ────────────────────────────────────────────────────
evaluator = Evaluator(cfg, class_names=dataset.class_names)
evaluator.accuracy_vs_snr(results_by_size)
evaluator.accuracy_vs_samples(results_by_fraction, img_size=128)

In [ ]:
# ── Ячейка 18: Матрицы ошибок ─────────────────────────────────────────────
trainer = Trainer(cfg)
train_128, test_128 = dataset.get_loaders(img_size=128)
centroids    = dataset.compute_centroids(train_128)
final_cnn    = trainer.train_cnn(EyeCNN(128, cfg.num_classes), train_128)
final_opt    = trainer.fit_optimal(centroids)

for snr_db in [20.0, 0.0]:
    lbl = "мало_шума" if snr_db == 20.0 else "много_шума"
    evaluator.plot_confusion_matrix(final_cnn, test_128, snr_db, f"CNN_SNR{snr_db}dB_{lbl}")
    evaluator.plot_confusion_matrix(final_opt, test_128, snr_db, f"Optimal_SNR{snr_db}dB_{lbl}")

In [ ]:
# ── Ячейка 19: Сохранение на Google Drive ────────────────────────────────
evaluator.save_to_drive()
print("Готово! Все файлы:")
from pathlib import Path
for f in sorted(Path("results").rglob("*.png")):
    print(f"  {f}")

In [ ]:
from PIL import Image
from google.colab import files
import torch

# Загружаем своё изображение
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# Препроцессинг — тот же что и для тест-данных
from torchvision import transforms
tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

img   = Image.open(img_path).convert("RGB")
x     = tf(img).unsqueeze(0).to(cfg.device)  # добавляем batch dimension

# Предсказание CNN
final_cnn.eval()
with torch.no_grad():
    logits = final_cnn(x)
    probs  = torch.softmax(logits, dim=1)[0]
    pred   = probs.argmax().item()

print(f"Класс: {dataset.class_names[pred]}")
print("Вероятности:")
for name, p in zip(dataset.class_names, probs):
    print(f"  {name:25s}: {p:.1%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_loss, label='Train')
ax1.plot(val_loss, label='Val')
ax1.set_title('Loss')
ax1.legend()
ax2.plot(train_acc, label='Train')
ax2.plot(val_acc, label='Val')
ax2.set_title('Accuracy')
ax2.legend()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')